# Structure Analysis

Run the TF/TG structure analysis from the current `.pt` models and droplet-union binary data files.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "inflow").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "inflow"))

import analyze_structure_script_generalized as structure
import get_st_funcs_generalized as gs


In [ ]:
tissue = "Pancreas"
ages = [3, 24]
lambda_by_age = {3: 0.7, 24: 0.4}
theta_threshold_by_age = {3: 0.02, 24: 0.05}

data_dir = repo_root / "data"
models_dir = repo_root / "outputs/models/final"
out_dir = repo_root / "outputs/structure"
out_dir.mkdir(parents=True, exist_ok=True)

age_a, age_b = sorted(ages)


In [ ]:
theta_tf_a = structure.load_theta(models_dir, tissue, age_a, "TF", lambda_by_age[age_a], theta_threshold_by_age[age_a])
theta_tf_b = structure.load_theta(models_dir, tissue, age_b, "TF", lambda_by_age[age_b], theta_threshold_by_age[age_b])
theta_tg_a = structure.load_theta(models_dir, tissue, age_a, "TG", lambda_by_age[age_a], theta_threshold_by_age[age_a])
theta_tg_b = structure.load_theta(models_dir, tissue, age_b, "TG", lambda_by_age[age_b], theta_threshold_by_age[age_b])

tf_data_path = gs.data_path(data_dir, tissue, age_a, "TF")
tf_data = np.load(tf_data_path)
names_tf = structure.load_names_tf(data_dir, tissue, theta_tf_a.shape[0])

print("TF data:", tf_data_path, tf_data.shape)
print(f"theta_tf[{age_a}] shape=", theta_tf_a.shape)
print(f"theta_tf[{age_b}] shape=", theta_tf_b.shape)
print(f"theta_tg[{age_a}] shape=", theta_tg_a.shape)
print(f"theta_tg[{age_b}] shape=", theta_tg_b.shape)


In [ ]:
timings = structure.calc_all_structs(
    t=tissue,
    theta_tf_a=theta_tf_a,
    theta_tf_b=theta_tf_b,
    theta_tg_a=theta_tg_a,
    theta_tg_b=theta_tg_b,
    age_a=age_a,
    age_b=age_b,
    names_tf=names_tf,
    out_dir=out_dir,
    factor=0.0,
    return_timings=True,
)


In [ ]:
timing_df = pd.DataFrame(
    {"step": list(timings.keys()), "seconds": list(timings.values())}
).sort_values("seconds", ascending=False)
display(timing_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(timing_df["step"], timing_df["seconds"], color="#6f8fa6")
ax.invert_yaxis()
ax.set_xlabel("Seconds")
ax.set_title(f"Structure runtime profile: {tissue}")
plt.tight_layout()
plt.show()


In [ ]:
ifl_counts = pd.read_csv(out_dir / f"IFL_counts_df_tf_{tissue}.csv")
iffl_counts = pd.read_csv(out_dir / f"IFFL_counts_df_tf_{tissue}.csv")
two_node_counts = pd.read_csv(out_dir / f"2_node_feedback_counts_df_{tissue}.csv")

display(ifl_counts.sort_values(["age", "pattern"]).reset_index(drop=True).head(20))
display(iffl_counts.sort_values(["age", "pattern"]).reset_index(drop=True).head(20))
display(two_node_counts.sort_values(["age", "pattern"]).reset_index(drop=True).head(20))


In [ ]:
sp_a = np.count_nonzero(theta_tf_a) / theta_tf_a.size
sp_b = np.count_nonzero(theta_tf_b) / theta_tf_b.size
if sp_a < sp_b:
    sparse_age, dense_age = age_a, age_b
else:
    sparse_age, dense_age = age_b, age_a

out_deg_dense = np.load(out_dir / f"out_deg_{tissue}_{dense_age}m_tf_corrected_sp_sp_age_{sparse_age}m.npy")
out_deg_sparse = np.load(out_dir / f"out_deg_{tissue}_{sparse_age}m_tf_corrected_sp_dense_age_{dense_age}m.npy")
null_out_deg = np.load(out_dir / f"out_deg_{tissue}_null_tf.npy")

fig, ax = plt.subplots(figsize=(6, 5))
ax.hist(out_deg_dense.mean(axis=0), bins=30, alpha=0.6, label=f"{dense_age}m sparsified")
ax.hist(out_deg_sparse, bins=30, alpha=0.6, label=f"{sparse_age}m")
ax.hist(null_out_deg.mean(axis=0), bins=30, alpha=0.4, label="null")
ax.set_xlabel("Out degree")
ax.set_ylabel("Count")
ax.legend()
ax.set_title(f"{tissue} TF out-degree summary")
plt.show()


CLI equivalent:

```bash
python inflow/analyze_structure_script_generalized.py \
  --tissue Pancreas \
  --ages 3 24 \
  --models-dir outputs/models/final \
  --data-dir data \
  --out-dir outputs/structure \
  --lambda-by-age 3:0.7,24:0.4 \
  --theta-threshold-by-age 3:0.02,24:0.05
```